# Bakery Order Management with LangGraph

A deterministic bakery order workflow built with Python and LangGraph. It uses hard-coded prices, inventory, rules, and reply text. **There are no LLM calls and no API keys.**

Run the cells from top to bottom. The final cell invokes the graph on six example orders.

In [ ]:
%pip install -q langgraph

## 1. Define the order state and bakery data

`OrderState` tracks the supplied order, decisions made by each node, and every node visited. Inventory is checked per order and is not mutated between examples.

In [ ]:
from typing import TypedDict


class OrderState(TypedDict):
    # Input fields
    item: str
    quantity: int
    day: str
    city: str

    # Output and workflow fields
    status: str
    product_actually_sold: str
    delivery: str
    delivery_fee: float
    total: float
    message: str
    steps: list[str]

prices = {
    "challah": 18,
    "babka": 45,
    "rugelach": 30,
    "sourdough": 25,
}

inventory = {
    "challah": 10,
    "babka": 0,
    "rugelach": 3,
    "sourdough": 4,
}

# Replacement priority, cycled from the requested product.
substitution_order = ["babka", "rugelach", "sourdough", "challah"]

## 2. Add the graph nodes

Each node returns only the fields it updates. `steps` is extended in execution order.

In [ ]:
def validate_order(state: OrderState) -> dict:
    """Validate the requested product and quantity."""
    updates = {"steps": state["steps"] + ["validate_order"]}
    if state["quantity"] <= 0 or state["item"] not in prices:
        updates.update({
            "status": "rejected",
            "message": "Politely: couldn't understand order",
        })
    else:
        updates.update({
            "status": "accepted",
            "product_actually_sold": state["item"],
        })
    return updates


def inventory_and_subs(state: OrderState) -> dict:
    """Find enough stock for the full quantity, trying up to two substitutes."""
    steps = state["steps"] + ["inventory_and_subs"]
    quantity = state["quantity"]
    requested_product = state["product_actually_sold"]

    if inventory[requested_product] >= quantity:
        return {"product_actually_sold": requested_product, "steps": steps}

    start = substitution_order.index(requested_product)
    alternatives = [
        substitution_order[(start + offset) % len(substitution_order)]
        for offset in range(1, len(substitution_order))
    ]
    for candidate in alternatives[:2]:
        if inventory[candidate] >= quantity:
            return {"product_actually_sold": candidate, "steps": steps}

    return {
        "status": "rejected",
        "message": "Apologize, out of stock and cancelling order",
        "steps": steps,
    }


def delivery_calc(state: OrderState) -> dict:
    if state["city"] == "Jerusalem":
        delivery, delivery_fee = "delivery", 20
    else:
        delivery, delivery_fee = "pickup", 0
    return {
        "delivery": delivery,
        "delivery_fee": delivery_fee,
        "steps": state["steps"] + ["delivery_calc"],
    }


def discount_calc(state: OrderState) -> dict:
    base_total = state["quantity"] * prices[state["product_actually_sold"]]
    if base_total > 150 and state["day"] != "Friday":
        base_total *= 0.85
    return {
        "total": round(base_total + state["delivery_fee"], 2),
        "steps": state["steps"] + ["discount_calc"],
    }


def format_reply(state: OrderState) -> dict:
    requested = state["item"]
    sold = state["product_actually_sold"]
    if sold == requested:
        product_note = f"your {state['quantity']} {sold} order"
    else:
        product_note = (
            f"your {state['quantity']} {sold} instead of {requested} "
            "because it was the available substitute"
        )
    if state["delivery"] == "delivery":
        delivery_note = "We will deliver it to you in Jerusalem."
    else:
        delivery_note = "Your order will be ready for pickup."
    message = (
        f"Thank you for choosing our bakery! We have confirmed {product_note}. "
        f"{delivery_note} Your total is ₪{state['total']:.2f}. We look forward to serving you!"
    )
    return {"message": message, "steps": state["steps"] + ["format_reply"]}

## 3. Build and draw the LangGraph workflow

Rejected orders stop after validation or inventory checks. Accepted orders continue through delivery, pricing, and reply formatting.

In [ ]:
from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


def after_validation(state: OrderState) -> str:
    return "inventory_and_subs" if state["status"] == "accepted" else END


def after_inventory(state: OrderState) -> str:
    return "delivery_calc" if state["status"] == "accepted" else END


builder = StateGraph(OrderState)
builder.add_node("validate_order", validate_order)
builder.add_node("inventory_and_subs", inventory_and_subs)
builder.add_node("delivery_calc", delivery_calc)
builder.add_node("discount_calc", discount_calc)
builder.add_node("format_reply", format_reply)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges(
    "validate_order", after_validation,
    {"inventory_and_subs": "inventory_and_subs", END: END},
)
builder.add_conditional_edges(
    "inventory_and_subs", after_inventory,
    {"delivery_calc": "delivery_calc", END: END},
)
builder.add_edge("delivery_calc", "discount_calc")
builder.add_edge("discount_calc", "format_reply")
builder.add_edge("format_reply", END)
graph = builder.compile()

# Render the compiled graph as a Mermaid PNG.
display(Image(graph.get_graph().draw_mermaid_png()))

## 4. Invoke the graph on six example orders

Each result includes the six requested output fields. `steps` shows which nodes ran for that case.

In [ ]:
from pprint import pprint


def run_order(order: dict) -> dict:
    initial_state: OrderState = {
        **order,
        "status": "",
        "product_actually_sold": "",
        "delivery": "",
        "delivery_fee": 0,
        "total": 0,
        "message": "",
        "steps": [],
    }
    result = graph.invoke(initial_state)
    output_keys = [
        "status", "product_actually_sold", "delivery",
        "total", "message", "steps",
    ]
    return {key: result[key] for key in output_keys}


test_cases = [
    ("Normal order", {"item": "challah", "quantity": 2, "day": "Monday", "city": "Jerusalem"}),
    ("Rejected invalid input", {"item": "croissant", "quantity": 1, "day": "Tuesday", "city": "Jerusalem"}),
    ("Order requiring substitution", {"item": "babka", "quantity": 2, "day": "Wednesday", "city": "Jerusalem"}),
    ("Cancelled because substitutes are unavailable", {"item": "rugelach", "quantity": 11, "day": "Thursday", "city": "Tel Aviv"}),
    ("Order with discount", {"item": "challah", "quantity": 10, "day": "Monday", "city": "Jerusalem"}),
    ("Outside Jerusalem (pickup)", {"item": "rugelach", "quantity": 2, "day": "Friday", "city": "Haifa"}),
]

for name, order in test_cases:
    print(f"\n{'=' * 72}\n{name}\n{'=' * 72}")
    pprint(run_order(order), sort_dicts=False, width=100)